# Multi-Agent Validation Demo

This notebook creates two temporary agents:

1. A RAG answer agent
2. A validation agent

The notebook reads questions from the synthetic questionnaire CSV.

For each selected question, the answer agent writes a response using Azure AI Search. The validation agent then reviews whether the answer is supported by the indexed documents.

Both temporary agents are deleted at the end.

## Imports and settings

In [1]:
from pathlib import Path
from dotenv import load_dotenv
import os
import csv

from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import ConnectionType

from azure.ai.agents import AgentsClient
from azure.ai.agents.models import AzureAISearchTool, MessageRole


# The notebook is inside walkthrough/, so .env is one folder above it.
load_dotenv(dotenv_path=Path("../.env"))

AIPROJECT_ENDPOINT = os.getenv("AIPROJECT_ENDPOINT")
CHAT_MODEL = os.getenv("CHAT_MODEL")
AZURE_AI_SEARCH_INDEX = os.getenv("AZURE_AI_SEARCH_INDEX")

QUESTIONNAIRE_PATH = Path("../synthetic_data/sample_security_questionnaire.csv")

# Change this number to test more or fewer questions from the CSV.
N_QUESTIONS = 3

# Low temperature keeps the questionnaire-style answers more consistent.
TEMPERATURE = 0.2

print("AIPROJECT_ENDPOINT set:", bool(AIPROJECT_ENDPOINT))
print("CHAT_MODEL:", CHAT_MODEL)
print("AZURE_AI_SEARCH_INDEX:", AZURE_AI_SEARCH_INDEX)
print("QUESTIONNAIRE_PATH:", QUESTIONNAIRE_PATH)
print("N_QUESTIONS:", N_QUESTIONS)
print("TEMPERATURE:", TEMPERATURE)

AIPROJECT_ENDPOINT set: True
CHAT_MODEL: gpt-4.1-mini
AZURE_AI_SEARCH_INDEX: trust-response-agent
QUESTIONNAIRE_PATH: ../synthetic_data/sample_security_questionnaire.csv
N_QUESTIONS: 3
TEMPERATURE: 0.2


## Create clients and Search tool

In [2]:
credential = DefaultAzureCredential()

project_client = AIProjectClient(
    endpoint=AIPROJECT_ENDPOINT,
    credential=credential
)

agents_client = AgentsClient(
    endpoint=AIPROJECT_ENDPOINT,
    credential=credential
)

search_connection = project_client.connections.get_default(
    connection_type=ConnectionType.AZURE_AI_SEARCH
)

ai_search = AzureAISearchTool(
    index_connection_id=search_connection.id,
    index_name=AZURE_AI_SEARCH_INDEX,
    top_k=5
)

print("Clients created.")
print("Search connection found:", bool(search_connection.id))
print("Azure AI Search tool created.")

Clients created.
Search connection found: True
Azure AI Search tool created.


## Load questions from the CSV

In [3]:
questions = []

with open(QUESTIONNAIRE_PATH, newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)

    for row in reader:
        questions.append(row)

questions_to_run = questions[:N_QUESTIONS]

print("Total questions in CSV:", len(questions))
print("Questions selected for this run:", len(questions_to_run))

for row in questions_to_run:
    print(row["question_id"], "-", row["category"], "-", row["question"])

Total questions in CSV: 20
Questions selected for this run: 3
Q001 - Access Control - Does AcmeCloud Analytics require MFA for employees and administrators?
Q002 - Access Control - How often is production access reviewed?
Q003 - Access Control - Are shared employee accounts permitted for standard business operations?


## Create temporary agents

In [4]:
answer_agent = agents_client.create_agent(
    model=CHAT_MODEL,
    name="temporary-rag-answer-agent",
    instructions=(
        "You answer trust and security questionnaire questions using the connected Azure AI Search index. "
        "Use the indexed documents when they contain the answer. "
        "If the indexed documents do not specify the answer, say that the evidence does not specify it."
    ),
    tools=ai_search.definitions,
    tool_resources=ai_search.resources,
    temperature=TEMPERATURE
)

validation_agent = agents_client.create_agent(
    model=CHAT_MODEL,
    name="temporary-validation-agent",
    instructions=(
        "You review answers to trust and security questionnaire questions. "
        "Use the connected Azure AI Search index to check whether the answer is supported by the indexed documents. "
        "Be strict. If the evidence is missing or incomplete, say that human review is needed."
    ),
    tools=ai_search.definitions,
    tool_resources=ai_search.resources,
    temperature=TEMPERATURE
)

print("RAG answer agent created.")
print("Validation agent created.")

RAG answer agent created.
Validation agent created.


## Ask and validate N questions

In [5]:
results = []

for row in questions_to_run:
    question_id = row["question_id"]
    category = row["category"]
    question = row["question"]

    print("=" * 100)
    print("Question ID:", question_id)
    print("Category:", category)
    print("Question:", question)

    answer_thread = agents_client.threads.create()

    agents_client.messages.create(
        thread_id=answer_thread.id,
        role="user",
        content=question
    )

    answer_run = agents_client.runs.create_and_process(
        thread_id=answer_thread.id,
        agent_id=answer_agent.id
    )

    answer_message = agents_client.messages.get_last_message_text_by_role(
        thread_id=answer_thread.id,
        role=MessageRole.AGENT
    )

    answer_text = answer_message.text.value

    print("\nAnswer run status:", answer_run.status)
    print("\nRAG answer:")
    print(answer_text)

    validation_prompt = f"""
    Question:
    {question}
    
    Answer:
    {answer_text}
    
    Review the answer against the indexed documents.
    
    Use these rules:
    - "Supported by evidence" means the indexed documents support the information requested by the original question.
    - If the original question asks for information that is not present in the indexed documents, set Supported by evidence to No.
    - If the answer safely says the evidence does not specify the information, mention that in the Reason.
    - If the original question asks for information that is missing, out of scope, future-dependent, customer-specific, legal, contractual, or operationally sensitive, set Human review needed to Yes.
    - Do not mark Supported by evidence as Yes only because the answer safely says the evidence does not specify it.
    
    Use this format:
    Supported by evidence: Yes / No / Partially
    Reason:
    Missing or uncertain information:
    Human review needed: Yes / No
    """

    validation_thread = agents_client.threads.create()

    agents_client.messages.create(
        thread_id=validation_thread.id,
        role="user",
        content=validation_prompt
    )

    validation_run = agents_client.runs.create_and_process(
        thread_id=validation_thread.id,
        agent_id=validation_agent.id
    )

    validation_message = agents_client.messages.get_last_message_text_by_role(
        thread_id=validation_thread.id,
        role=MessageRole.AGENT
    )

    validation_text = validation_message.text.value

    print("\nValidation run status:", validation_run.status)
    print("\nValidation review:")
    print(validation_text)

    results.append({
        "question_id": question_id,
        "category": category,
        "question": question,
        "answer": answer_text,
        "validation": validation_text
    })

print("=" * 100)
print("Finished questions:", len(results))

Question ID: Q001
Category: Access Control
Question: Does AcmeCloud Analytics require MFA for employees and administrators?

Answer run status: RunStatus.COMPLETED

RAG answer:
Yes, AcmeCloud Analytics requires multi-factor authentication (MFA) for all employees and administrators before accessing company-managed systems. This MFA requirement applies to email, internal collaboration tools, code repositories, deployment consoles, production monitoring dashboards, and administrative interfaces. Password-only access is not considered sufficient for internal systems. Temporary exceptions to MFA require written approval and have a maximum duration of seven calendar days. Shared employee accounts are not permitted; named accounts must be used to associate activity with individual users, and administrative access also requires MFA as part of role-based access controls【3:0†Access Control Policy】,【3:2†Encryption Policy】.

Validation run status: RunStatus.COMPLETED

Validation review:
Supported 

## Delete temporary agents

In [6]:
if "answer_agent" in globals() and answer_agent is not None:
    agents_client.delete_agent(answer_agent.id)
    print("Deleted RAG answer agent.")

if "validation_agent" in globals() and validation_agent is not None:
    agents_client.delete_agent(validation_agent.id)
    print("Deleted validation agent.")

Deleted RAG answer agent.
Deleted validation agent.
